## AY25-26 Term 2 CS421 Group Project

In [2]:
%reload_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis, entropy
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    f1_score
)
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
import mlflow

from features.feature_pipeline import BaselineFeatureBuilder, InteractionFeatureBuilder
first_batch_data=np.load("../input/training_batch_with_labels.npz")
second_batch_data=np.load("../input/first_batch_with_labels.npz")


In [3]:
X = np.vstack([first_batch_data['X'], second_batch_data['X']])
y = np.concatenate([first_batch_data['y'], second_batch_data['y']], axis=0)
print("# of interactions:", X.shape[0])
print("# of anomalous and normal users:", np.count_nonzero(y==1), np.count_nonzero(y==0))


df=pd.DataFrame(X)
labels=pd.DataFrame(y)
df.rename(columns={0:"user",1:"item",2:"rating"},inplace=True)
print("# of items:", df['item'].unique().shape[0])

# of interactions: 344839
# of anomalous and normal users: 200 2000
# of items: 996


In [4]:
df.head(100)
print(df.shape)

(344839, 3)


In [5]:
labels.rename(columns={0:"user",1:"label"},inplace=True)
labels


,user,label
0,100,0
1,101,0
2,102,0
3,103,0
4,104,0
...,...,...
2195,3595,0
2196,3596,1
2197,3597,0
2198,3598,0


## Feature Engineering

The objective is to derive user-level behavioural features from the interaction dataset (user, item, rating) so that the model can learn behavioural patterns that distinguish anomalous users from normal users.

Since anomaly detection occurs at the user level, all features must summarize a user's rating behaviour across items.

Derive BaseLine user level features.

In [6]:
bf = BaselineFeatureBuilder()
user_features_baseline = bf.transform(df)
user_features_baseline.head()
print(user_features_baseline.shape)


/Users/Kathir/Documents/Coursework/machine_learning/project/src/features/feature_pipeline.py:29: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  user_rating_skew=("rating", lambda x: skew(x)),
/Users/Kathir/Documents/Coursework/machine_learning/project/src/features/feature_pipeline.py:30: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  user_rating_kurt=("rating", lambda x: kurtosis(x)),


(2200, 17)


## Train XGBoost with Item and Interaction level features

In [7]:
users = user_features_baseline["user"]
y = labels['label']

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
roc_scores = []
pr_scores = []
f1_scores = []
feature_importance_list = []
feature_names = None


model_params= {
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.01,
    "subsample": 0.7,
    "colsample_bytree": 0.7,
    "scale_pos_weight": 17,
    "random_state": 42
}

for fold, (train_idx, val_idx) in enumerate(kf.split(users, y)):

    print(f"\nFold {fold+1}")

    train_users = users.iloc[train_idx]
    val_users = users.iloc[val_idx]

    # ---------- SPLIT RAW DATA ----------
    train_df = df[df["user"].isin(train_users)].copy()
    val_df   = df[df["user"].isin(val_users)].copy()

    
    ibf = InteractionFeatureBuilder()
    ibf.fit(train_df)
    interaction_features_train = ibf.transform(train_df)
    interaction_features_val = ibf.transform(val_df)

    # merge with baseline features
    train_features = user_features_baseline[
        user_features_baseline["user"].isin(train_users)
    ].merge(interaction_features_train, on="user", how="left")

    val_features = user_features_baseline[
        user_features_baseline["user"].isin(val_users)
    ].merge(interaction_features_val, on="user", how="left")

    X_train = train_features.drop(columns=["user"])
    y_train = y[train_idx]

    if feature_names is None:
        feature_names = X_train.columns

    X_val = val_features.drop(columns=["user"])
    y_val = y[val_idx]


    model = xgb.XGBClassifier(**model_params)

    model.fit(X_train, y_train)

    y_prob = model.predict_proba(X_val)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)  

    f1 = f1_score(y_val, y_pred)
    roc = roc_auc_score(y_val, y_prob)
    pr = average_precision_score(y_val, y_prob)

    f1_scores.append(f1)
    roc_scores.append(roc)
    pr_scores.append(pr)
    feature_importance_list.append(model.feature_importances_)

    print(f"ROC-AUC: {roc:.4f}, PR-AUC: {pr:.4f}, F1: {f1:.4f}")
    print(f"ROC-AUC: {roc:.4f}, PR-AUC: {pr:.4f}")


print("\nFinal Results:")
print("Mean ROC-AUC:", sum(roc_scores)/len(roc_scores))
print("Mean PR-AUC:", sum(pr_scores)/len(pr_scores))
print("Mean F1:", sum(f1_scores)/len(f1_scores))

mean_importance = np.mean(feature_importance_list, axis=0)

feature_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": mean_importance
}).sort_values(by="importance", ascending=False)
feature_importance_df



Fold 1
ROC-AUC: 0.8428, PR-AUC: 0.5858, F1: 0.5000
ROC-AUC: 0.8428, PR-AUC: 0.5858

Fold 2
ROC-AUC: 0.8939, PR-AUC: 0.6966, F1: 0.5773
ROC-AUC: 0.8939, PR-AUC: 0.6966

Fold 3
ROC-AUC: 0.9481, PR-AUC: 0.7331, F1: 0.5684
ROC-AUC: 0.9481, PR-AUC: 0.7331

Fold 4
ROC-AUC: 0.9009, PR-AUC: 0.7095, F1: 0.5524
ROC-AUC: 0.9009, PR-AUC: 0.7095

Fold 5
ROC-AUC: 0.9229, PR-AUC: 0.7629, F1: 0.5872
ROC-AUC: 0.9229, PR-AUC: 0.7629

Final Results:
Mean ROC-AUC: 0.9017
Mean PR-AUC: 0.6975700159700295
Mean F1: 0.5570555111888299


,feature,importance
9,rating_1_pct,0.085073
4,min_rating,0.078053
19,std_normalized_deviation,0.066642
24,negative_dev_ratio,0.057873
14,rating_entropy,0.055441
8,rating_0_pct,0.044249
20,mean_abs_deviation,0.044014
17,std_rating_deviation,0.042985
23,p90_abs_deviation,0.040122
13,user_rating_kurt,0.038983


## Training on entire dataset

In [169]:
# Use full data
train_df = df.copy()
# compute item stats again
ibf_full_data = InteractionFeatureBuilder()
ibf_full_data.fit(train_df)
interaction_features = ibf_full_data.transform(train_df)

# merge with user_level_features_baseline
full_features = user_features_baseline.merge(
    interaction_features,
    on="user",
    how="left"
).fillna(0)

X = full_features.drop(columns=["user"])
y = labels['label']

print(f'Shape of labels: {y.shape}')
print(f'Shape of full features: {X.shape}')


model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=10,
    random_state=42
)

model.fit(X, y)

#  Load test df
test_data = np.load('../input/second_batch.npz')
test_data = test_data['X']
test_df = pd.DataFrame(
    test_data,
    columns=["user","item","rating"]
)

test_bf = BaselineFeatureBuilder()
user_features_baseline_test = test_bf.transform(test_df)
print(f'{user_features_baseline_test.shape[0]} new users in test data!!!!')

# Apply SAME feature logic using TRAIN statistics
test_interaction_features = ibf_full_data.transform(test_df)
test_features = user_features_baseline_test.merge(
    test_interaction_features,
    on="user",
    how="left"
).fillna(0)


# batch1_test = test_features.drop(columns=["user"])
batch2_test = test_features.drop(columns=["user"])


# batch1_test_probs = model.predict_proba(batch1_test)[:, 1]
batch2_test_probs = model.predict_proba(batch2_test)[:, 1]
print(batch2_test_probs[:5])
np.savez('submission.npz', predictions=batch2_test_probs)

Shape of labels: (2200,)
Shape of full features: (2200, 26)


/Users/Kathir/Documents/Coursework/machine_learning/project/src/features/feature_pipeline.py:5: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  class BaselineFeatureBuilder:


860 new users in test data!!!!
[0.05213026 0.00106831 0.09882795 0.14121674 0.00031414]
